In [ ]:
!pip install git+https://github.com/KaiyangZhou/deep-person-reid.git

  Cloning https://github.com/KaiyangZhou/deep-person-reid.git to /tmp/pip-req-build-pyjek5io
  Running command git clone --filter=blob:none --quiet https://github.com/KaiyangZhou/deep-person-reid.git /tmp/pip-req-build-pyjek5io
  Resolved https://github.com/KaiyangZhou/deep-person-reid.git to commit 566a56a2cb255f59ba75aa817032621784df546a
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 5.9 MB/s eta 0:00:00
  Created wheel for torchreid: filename=torchreid-1.4.0-cp312-cp312-linux_x86_64.whl size=714572 sha256=5dcc89d2eea5b06598490d70c31d71f8e25ac2b5

In [ ]:
!pip install gdown

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/modellerim'
os.makedirs(DRIVE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from __future__ import absolute_import
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchreid
from torchreid.data import transforms as T
from torchreid.data.sampler import RandomIdentitySampler
import torch.optim as optim
import numpy
from torchreid.engine import Engine

In [ ]:

!mkdir -p /content/reid-data/market1501
%cd /content/reid-data/market1501

!gdown --id 0B8-rUzbwVRk0c054eEozWG9COHM
!unzip Market-1501-v15.09.15.zip
%cd ..
%cd ..

!mkdir -p /content/reid-data/dukemtmc-reid
%cd /content/reid-data/dukemtmc-reid
!gdown --id 1jjE85dRCMOgRtvJ5RQV9-Afs-2_5dY3O

!unzip DukeMTMC-reID.zip

%cd ..
%cd ..

/content/reid-data/msmt17
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1n_0v0L6dHTKX-0161a-nOF2E10S7C-2z

but Gdown can't. Please check connections and permissions.
unzip:  cannot find or open MSMT17_V2.zip, MSMT17_V2.zip.zip or MSMT17_V2.zip.ZIP.
/content/reid-data
/content


In [ ]:
DATA_DIR = 'reid-data'
IMG_HEIGHT = 384
IMG_WIDTH = 128
TRAIN_BATCH_SIZE = 128
NUM_INSTANCES = 8
WORKERS = 8

print(f"torchreid version: {torchreid.__version__}")

datamanager = torchreid.data.ImageDataManager(
    root=DATA_DIR,
    sources='market1501',
    targets='market1501',
    height=IMG_HEIGHT,
    width=IMG_WIDTH,
    batch_size_train=TRAIN_BATCH_SIZE,
    batch_size_test=100,
    num_instances=NUM_INSTANCES,
    train_sampler='RandomIdentitySampler',
    transforms=['random_flip', 'color_jitter', 'random_erase'],
    workers=WORKERS
)

train_loader = datamanager.train_loader
test_loader = datamanager.test_loader
num_classes_train = datamanager.num_train_pids


testdatamanager = torchreid.data.ImageDataManager(
    root=DATA_DIR,
    sources='dukemtmcreid',
    targets='dukemtmcreid',
    height=IMG_HEIGHT,
    width=IMG_WIDTH,
    batch_size_train=TRAIN_BATCH_SIZE,
    batch_size_test=100,
    num_instances=NUM_INSTANCES,
    train_sampler='RandomIdentitySampler',
    transforms=['random_flip', 'color_jitter', 'random_erase'],
    workers=WORKERS
)



torchreid version: 1.4.0
Building train transforms ...
+ resize to 384x128
+ random flip
+ color jitter
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
+ random erase
Building test transforms ...
+ resize to 384x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded Market1501
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  

In [ ]:
class CenterLoss(nn.Module):
    """Center loss.

    Reference:
    Wen et al. A Discriminative Feature Learning Approach for Deep Face Recognition. ECCV 2016.

    Args:
        num_classes (int): number of classes.
        feat_dim (int): feature dimension.
    """

    def __init__(self, num_classes=751, feat_dim=2048, use_gpu=True):
        super(CenterLoss, self).__init__()
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.use_gpu = use_gpu

        if self.use_gpu:
            self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim).cuda())
        else:
            self.centers = nn.Parameter(torch.randn(self.num_classes, self.feat_dim))

    def forward(self, x, labels):
        """
        Args:
            x: feature matrix with shape (batch_size, feat_dim).
            labels: ground truth labels with shape (num_classes).
        """
        assert x.size(0) == labels.size(0), "features.size(0) is not equal to labels.size(0)"

        batch_size = x.size(0)
        distmat = torch.pow(x, 2).sum(dim=1, keepdim=True).expand(batch_size, self.num_classes) + \
                  torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(self.num_classes, batch_size).t()
        distmat.addmm_( x, self.centers.t(),beta=1,alpha= -2,)

        classes = torch.arange(self.num_classes).long()
        if self.use_gpu: classes = classes.cuda()
        labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
        mask = labels.eq(classes.expand(batch_size, self.num_classes))

        dist = []
        for i in range(batch_size):
            value = distmat[i][mask[i]]
            value = value.clamp(min=1e-12, max=1e+12)  # for numerical stability
            dist.append(value)
        dist = torch.cat(dist)
        loss = dist.mean()
        return loss


from __future__ import absolute_import

import torch
from torch import nn
from torch.autograd import Variable


class TripletLoss(nn.Module):
    def __init__(self, margin=0):
        super(TripletLoss, self).__init__()
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, inputs, targets):
        n = inputs.size(0)
        # Compute pairwise distance, replace by the official when merged
        dist = torch.pow(inputs, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        dist.addmm_(inputs, inputs.t(),beta=1,alpha= -2 )
        dist = dist.clamp(min=1e-12).sqrt()  # for numerical stability
        # For each anchor, find the hardest positive and negative
        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap, dist_an = [], []
        for i in range(n):
            dist_ap.append(dist[i][mask[i]].max())
            dist_an.append(dist[i][mask[i] == 0].min())
        dist_ap = torch.stack(dist_ap, dim=0)
        dist_an = torch.stack(dist_an, dim=0)
        # Compute ranking hinge loss
        y = dist_an.data.new()
        y.resize_as_(dist_an.data)
        y.fill_(1)
        y = Variable(y)
        loss = self.ranking_loss(dist_an, dist_ap, y)
        prec = (dist_an.data > dist_ap.data).sum() * 1. / y.size(0)
        return loss, prec

class AMSoftmax(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.30):
        super(AMSoftmax, self).__init__()
        self.m = m
        self.s = s
        self.in_feats = in_features
        self.W = torch.nn.Parameter(torch.randn(in_features, out_features), requires_grad=True)
        self.ce = nn.CrossEntropyLoss()
        nn.init.xavier_normal_(self.W, gain=1)

    def forward(self, x, lb):
        assert x.size()[0] == lb.size()[0]
        assert x.size()[1] == self.in_feats
        x_norm = torch.norm(x, p=2, dim=1, keepdim=True).clamp(min=1e-12)
        x_norm = torch.div(x, x_norm)
        w_norm = torch.norm(self.W, p=2, dim=0, keepdim=True).clamp(min=1e-12)
        w_norm = torch.div(self.W, w_norm)
        costh = torch.mm(x_norm, w_norm)
        lb_view = lb.view(-1, 1)
        delt_costh = torch.zeros(costh.size(), device=x.device).scatter_(1, lb_view, self.m)
        costh_m = costh - delt_costh
        costh_m_s = self.s * costh_m
        return self.ce(costh_m_s,lb)



In [ ]:
MODEL_DIR = os.getcwd()
class MultiAttentionRe_id(nn.Module):
    def __init__(self,model_name ="osnet_ain_x1_0", pretrain_model = True,weights_pth_model = os.path.join(MODEL_DIR,"bestfor_all.pth"),pretrain_backbone = False,weights_pth_backbone=os.path.join(MODEL_DIR,"osnet.pth"), train_backbone=True, *args, **kwargs):
        super().__init__(*args, **kwargs)


        #Backbone
        self.backbone = torchreid.models.build_model(name=model_name,num_classes=2)
        if pretrain_backbone:
          torchreid.utils.load_pretrained_weights(self.backbone, weights_pth_backbone)
        for x in ["classifier","fc","global_avgpool"]:self.backbone.__delattr__(name=x)
        self.backbone.requires_grad_(train_backbone)

        self.outdim_of_backbone = 512

        self.training = True

        #PAM attention module
        self.in_chanel = self.outdim_of_backbone
        self.out_channel = self.in_chanel//16
        self.conv1_pam = nn.Conv2d(in_channels=self.in_chanel,out_channels=self.out_channel,kernel_size=3,stride=1,padding=1)
        self.conv2_pam = nn.Conv2d(in_channels=self.in_chanel,out_channels=self.out_channel,kernel_size=1,stride=1,padding=0)
        self.pam_softmax = nn.Softmax2d()
        self.pam_batchnorm = nn.BatchNorm2d(num_features=self.in_chanel)

        #ECA chanel attention module
        self.global_avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.conv3_eca = nn.Conv1d(in_channels=self.outdim_of_backbone,out_channels=self.outdim_of_backbone,kernel_size=3,stride=1,padding=1)
        self.eca_sigmoid = nn.Sigmoid()

        #Feature Part
        self.feature_out_dims = self.outdim_of_backbone //2
        self.relu = nn.ReLU()


        self.feature_lin_list = nn.ModuleList()
        self.feature_conv_list = nn.ModuleList()
        # 2. 6 kez döngüye girin

        for _ in range(6):
            # 3. Her döngüde YENİ, BAĞIMSIZ bir Linear katman oluşturun

            conv = nn.Conv1d(
                in_channels=self.in_chanel,
                out_channels=self.feature_out_dims*2,
                kernel_size=1,stride=1,
                bias=False
            )
            conv1 = nn.Conv1d(
                in_channels=self.feature_out_dims*2,
                out_channels=self.feature_out_dims,
                kernel_size=1,stride=1,
                bias=False
            )
            conv = nn.Sequential(conv,self.relu,conv1)

            lineer = nn.Linear(
                in_features=self.in_chanel,
                out_features=self.feature_out_dims*2,
                bias=False
            )
            lineer55 = nn.Linear(
                in_features=self.feature_out_dims*2,
                out_features=self.feature_out_dims,
                bias=False
            )
            lineerS = nn.Sequential(lineer,self.relu,lineer55)

            # 4. Oluşturulan katmanı listeye ekleyin
            self.feature_lin_list.append(lineerS)
            self.feature_conv_list.append(conv)

        if pretrain_model:
          state = torch.load(weights_pth_model,map_location="cpu")
          self.load_state_dict(state)



    def Pam_module(self, x):
        # x, self.backbone.featuremaps(x)'ten gelen girdidir.
        B, C, H, W = x.shape
        N = H * W # N = H*W [cite: 171]

        # 1. Dal (Query - Q) [cite: 171]
        # (B, C, H, W) -> (B, C/16, H, W)
        q_features = self.conv1_pam(x)
        # (B, C/16, H, W) -> (B, C/16, N)
        Q = q_features.view(B, self.out_channel, N)

        # 2. Dal (Key - K) [cite: 171]
        # (B, C, H, W) -> (B, C/16, H, W)
        k_features = self.conv2_pam(x)
        # (B, C/16, H, W) -> (B, C/16, N)
        K = k_features.view(B, self.out_channel, N)
        # (B, C/16, N) -> (B, N, C/16) (Key Transpose)
        K_T = K.permute(0, 2, 1)

        # 3. Dal (Value - V) [cite: 172, 174]
        # V, orijinal 'x'in yeniden şekillendirilmiş halidir
        # (B, C, H, W) -> (B, C, N)
        V = x.view(B, C, N)

        # HATA DÜZELTME 3 (MMconv): (B, N, C/16) @ (B, C/16, N) -> (B, N, N)
        # Bu, S_raw (Ham Dikkat Haritası)
        S_raw = torch.bmm(K_T, Q)

        # SoftConv: (B, N, N)
        SoftConv = self.pam_softmax(S_raw)

        # HATA DÜZELTME 4 (MMX): V @ S
        # (B, C, N) @ (B, N, N) -> (B, C, N)
        # Bu, R_reshaped (Yeniden şekillendirilmiş R)
        MMX = torch.bmm(V, SoftConv)

        # HATA DÜZELTME 5 (BatchNorm Sırası)
        # (B, C, N) -> (B, C, H, W)
        # BatchNorm'dan ÖNCE R'yi 4D'ye geri döndür
        R = MMX.view(B, C, H, W)

        # Batchnorm
        Batchnorm = self.pam_batchnorm(R)

        # Artık (Residual) Toplama
        return x + Batchnorm

    def Eca_module(self, x):
        # x boyutu: (B, C, H, W)

        # 1. GAP [cite: 196]
        # (B, C, H, W) -> (B, C, 1, 1)
        Gap = self.global_avgpool(x)

        # 2. Transpose (ve Sıkıştırma - Squeeze)
        # (B, C, 1, 1) -> (B, C, 1)
        # .transpose(1, 2) metottur, .transpose değil
        # (B, C, 1,1) -> (B, C, 1)
        Gap_transposed = Gap.squeeze(-1)

        # 3. Conv1d
        # (B, C, 1) -> (B, C, 1)

        conv_eca = self.conv3_eca(Gap_transposed)


        # 5. Sigmoid [cite: 198]
        # (B, C, 1) -> (B, C, 1)
        sigma_eca = self.eca_sigmoid(conv_eca)

        # 6. Boyutları Eşleştirme (Unsqueeze)
        # Çarpma işlemi için (B, C, H, W) ile (B, C, 1, 1) gerekir
        # (B, C, 1) -> (B, C, 1, 1)
        sigma_eca_reshaped = sigma_eca.unsqueeze(-1)

        # 7. Eleman Çarpımı (HATA DÜZELTME 3)
        # @ (matmul) YERİNE * (element-wise)
        # (B, C, H, W) * (B, C, 1, 1) -> (B, C, H, W)
        return x * sigma_eca_reshaped




    def forward(self, x):
        # 1. Omurga (Backbone)
        # Girdi: (B, 3, 384, 128)
        # Çıktı: (B, C, H, W) -> (B, 512, 12, 4)
        x = self.backbone.featuremaps(x)

        # 2. PAM (Position Attention Module)
        # Çıktı: (B, 512, 12, 4)
        x = self.Pam_module(x)

        # 3. ECA (Efficient Channel Attention)
        # Bu, 'Refined Feature' (İyileştirilmiş Öznitelik) haritasıdır
        # Çıktı: (B, 512, 12, 4)
        refined_feature_map = self.Eca_module(x)

        # --- 4. Paralel Vektör Oluşturma (Global ve Lokal) ---

        # 4a. Global Vektör (Test/Metrik için)
        # (B, 512, 12, 4) -> (B, 512, 1, 1) -> (B, 512)
        global_vector = self.global_avgpool(refined_feature_map)
        global_vector = torch.flatten(global_vector, 1)

        # 4b. 6 Lokal Parça Vektörü
        B, C, H, W = refined_feature_map.shape
        part_height = H // 6 # 12 // 6 = 2

        part_vectors_list = [] # 6 adet (B, 512) vektörünü tutar

        for i in range(6):
            # Yatay şeridi al (B, 512, 2, 4)
            part_strip = refined_feature_map[:, :, i*part_height : (i+1)*part_height, :]

            # Şeride GAP uygula (B, 512, 1, 1)
            pooled_part = self.global_avgpool(part_strip)

            # Düzleştir (B, 512)
            flat_part = pooled_part.squeeze(-1)
            part_vectors_list.append(flat_part)

        # --- 5. İki Çıktıyı da Üret ---

        # Çıktı 1: part_logits (ID Kaybı için)
        # 6 adet (B, 512) vektörünü, 6 ayrı sınıflandırıcıya besle
        logits_list = None
        for i in range(6):
            part_vector = part_vectors_list[i]

            # 'self.part_heads' -> 6 sınıflandırıcıyı tutan nn.ModuleList
            conv = self.feature_conv_list[i]
            lineer = self.feature_lin_list[i]

            logitC = conv(part_vector) # (B, num_classes)
            l_in = part_vector.squeeze(-1)
            logitL = lineer(l_in)

            logits = logitC.squeeze(-1) + logitL
            logits = logits.squeeze(-1)
            logits = logits.unsqueeze(0) # (1, B, num_classes)
            if logits_list is None:
                logits_list = logits
            else:
                logits_list = torch.concat((logits_list,logits))

        # Çıktı 2: concatenated_feature (Test/Metrik Kaybı için)
        # [1 Global Vektör] + [6 Lokal Vektör]
        logits_list = logits_list.permute(1, 0, 2)
        concatenated_feature = torch.cat((global_vector, logits_list.flatten(start_dim=1)), dim=1)

        # --- 6. Nihai Çıktı ---

        # Modu (train/eval) kontrol et
        if self.training:
            # Eğitimde, hem Softmax (ID) kaybı için logits'e
            # hem de Metrik kayıplar (TriHard/Center) için 'concat' vektöre
            # ihtiyacımız var.
            return logits_list, concatenated_feature
        else:
            # Testte (eval), SADECE "gerçek hayat" (test) için
            # kullanılacak 'concatenated_feature' (parmak izi) döndürülür.
            return concatenated_feature


In [ ]:


def train():
  MODEL_DIR = os.getcwd()
  NUM_CLASSES_TRAIN = 751
  FEATURE_DIM = 512 + 6*256
  DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Setting up loss functions on {DEVICE} device.")


  loss_triplet = TripletLoss(margin=0.3).to(DEVICE)
  print("Triplet Loss initialized.")


  loss_center = CenterLoss(num_classes=NUM_CLASSES_TRAIN, feat_dim=FEATURE_DIM, use_gpu=torch.cuda.is_available()).to(DEVICE)
  print("Center Loss initialized.")

  loss_softmax = AMSoftmax(in_features=256, out_features=NUM_CLASSES_TRAIN, s=30.0, m=0.3).to(DEVICE)
  print("Softmax Loss initialized.")

  print("\nAll loss functions (ID, Triplet, Center) are set up successfully.")
  LEARNING_RATE_MODEL = 1e-1
  LEARNING_RATE_CENTER = 1e-3
  ALPHA = 0.6
  BETA = 0.9
  DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Eğitim {DEVICE} üzerinde çalışacak.")


  NUM_EPOCHS = 100
  LOG_INTERVAL = 10
  best_Rank1_acc = 0
  rank1 = []

  model = MultiAttentionRe_id(pretrain_backbone=True,train_backbone=True).to(DEVICE)
  EngineM = Engine(datamanager=datamanager,use_gpu=torch.cuda.is_available())
  EngineM.model = model
  model_optimizer = torch.optim.Adadelta(
        [{"params" : model.parameters()},
        {"params" : loss_softmax.parameters()}],
      lr=LEARNING_RATE_MODEL
  )
  center_optimizer = torch.optim.SGD(
      params = loss_center.parameters(),
      lr=LEARNING_RATE_CENTER,momentum=0.9
      )

  print("Gerçek Model ve 2 optimizer (model, center) hazırlandı.")
  print("Eğitim döngüsü başlıyor (test için 100 batch)...")



  print(f"Eğitim {NUM_EPOCHS} epoch için başlıyor...")

  for epoch in range(NUM_EPOCHS):
      model.train()
      epoch_total_loss = 0.0

      print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} Başladı ---")
      for batch_idx, data in enumerate(train_loader):
          images = data['img'].to(DEVICE)
          targets = data['pid'].to(DEVICE)
          part_features, concatenated_feature = model(images)
          model_optimizer.zero_grad()
          center_optimizer.zero_grad()
          part_features = part_features.permute(1, 0, 2)  # (B, 6, num_classes)
          total_id_loss = sum(loss_softmax(part_feat.squeeze(1), targets) for part_feat in part_features)
          total_triplet_loss = loss_triplet(concatenated_feature, targets)
          total_center_loss = loss_center(concatenated_feature, targets)
          L_total = total_id_loss + (torch.mul(BETA, total_triplet_loss[0])) + torch.mul(ALPHA, total_center_loss).reshape(1, 1)
          L_total = L_total.sum()
          L_total.backward()
          model_optimizer.step()
          center_optimizer.step()
          epoch_total_loss += L_total.item()
          if (batch_idx + 1) % LOG_INTERVAL == 0:
              print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}], Batch [{batch_idx+1}/{len(train_loader)}]: "
                    f"L_Total={L_total.item():.2f} "
                    f"(ID={total_id_loss.sum():.2f}, "
                    f"Triplet={total_triplet_loss[0]:.2f}, "
                    f"Center={total_center_loss:.2f})")
      print(f"--- Epoch {epoch+1} Özeti ---")
      print(f"  Ortalama Toplam Kayıp: {epoch_total_loss/len(train_loader):.4f}")
      print("-" * 30)
      model.eval()
      EngineM.model = model
      rank1_acc = EngineM.test("cosine")
      rank1.append(rank1_acc)
      if best_Rank1_acc <= rank1_acc:
        best_Rank1_acc = rank1_acc
        torch.save(model.state_dict(),os.path.join(DRIVE_DIR,"best1.pth"))


  print("\nEğitim tamamlandı.")

Eğitim cuda üzerinde çalışacak.
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_ain_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Successfully loaded pretrained weights from "./osnet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Gerçek Model ve 2 optimizer (model, center) hazırlandı.
Eğitim döngüsü başlıyor (test için 100 batch)...
Eğitim 100 epoch için başlıyor...

--- Epoch 1/100 Başladı ---
  Epoch [1/100], Batch [10/88]: L_Total=1304.85 (ID=103.40, Triplet=2.89, Center=1998.07)
  Epoch [1/100], Batch [20/88]: L_Total=1292.97 (ID=101.53, Triplet=1.66, Center=1983.24)
  Epoch [1/100], Batch [30/88]: L_Total=1303.98 (ID=101.92, Triplet=1.25, Center=2001.56)
  Epoch [1/100], Batch [40/88]: L_Total=1307.29 (ID=97.34, Triplet=1.28, Center=2014.66)
  Epoch [1/100], Batch [50/88]: L_Tot

In [ ]:
def test():

  DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = MultiAttentionRe_id(pretrain_model=True).to(DEVICE)
  model.eval()
  EngineM = Engine(datamanager=testdatamanager,use_gpu=torch.cuda.is_available())
  EngineM.model = model
  rank1_acc = EngineM.test( dist_metric="cosine",visrank=True,visrank_topk=10,save_dir=MODEL_DIR,use_metric_cuhk03=False,ranks=[1, 5, 10, 20],rerank=True)

test()

Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_ain_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
##### Evaluating dukemtmcreid (source) #####
Extracting features from query set ...
Done, obtained 2228-by-2048 matrix
Extracting features from gallery set ...
Done, obtained 17661-by-2048 matrix
Speed: 0.0266 sec/batch
Computing distance matrix with metric=cosine ...
Applying person re-ranking ...
Computing CMC and mAP ...
** Results **
mAP: 42.0%
CMC curve
Rank-1  : 54.8%
Rank-5  : 65.7%
Rank-10 : 70.2%
Rank-20 : 74.9%
# query: 2228
# gallery 17661
Visualizing top-10 ranks ...
- done 100/2228
- done 200/2228
- done 300/2228
- done 400/2228
- done 500/2228
- done 600/2228
- done 700/2228
- done 800/2228
- done 900/2228
- done 1000/2228
- done 1100/2228
- done 1200/2228
- done 1300/2228
- done 1400/2228
- done 1500/2228
- done 1600/2228
- done 1700/2228
- done